We want dialogue to appear in the format;
character: dialogue
And we want to extract
Character
Spell

We want to translate the character IDs into character names. Character names are stoed in the characters file. no shit.


In [3]:
import pandas as pd
import re

# Load dialogue
dialogue = pd.read_csv(
    "dialogue.csv",
    sep=",",
    encoding="latin1",
    quoting=3,
    on_bad_lines="skip"
)

# Load characters
characters = pd.read_csv("Characters.csv", encoding="cp1252")
character_map = dict(zip(characters["Character ID"], characters["Character Name"]))

# Keep character name + dialogue together
dialogue["Character Name"] = dialogue["Character ID"].map(character_map)
dialogue["Dialogue Text"] = dialogue["Dialogue"].fillna("")

# Import spell notebook output
spell_topics = pd.read_csv("spell_topic_matrix.csv")
print(spell_topics.head())


# Keep a clean spell lookup from the exported spell-topic matrix
spells_df = spell_topics.copy()

spells_df = spells_df.dropna(subset=["Incantation"])
spells_df["Incantation"] = (
    spells_df["Incantation"]
    .astype(str)
    .str.lower()
    .str.strip()
)
spells_df = spells_df.dropna(subset=["Incantation"])
spells_df["Incantation"] = spells_df["Incantation"].astype(str).str.lower().str.strip()

def normalize_for_matching(text: str) -> str:
    # Normalize punctuation/whitespace so multi-word incantations match consistently.
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def build_incantation_pattern(incantation: str) -> re.Pattern:
    tokens = [re.escape(t) for t in incantation.split() if t]
    if not tokens:
        # Matches nothing for empty tokens; keeps logic simple downstream.
        return re.compile(r"a^", flags=re.IGNORECASE)
    # Allow one or more spaces between words after normalization.
    pattern = r"\b" + r"\s+".join(tokens) + r"\b"
    return re.compile(pattern, flags=re.IGNORECASE)

# Deduplicate by normalized incantation and build regex patterns once
spells_df = spells_df.drop_duplicates(subset=["Incantation"]).reset_index(drop=True)
spell_patterns = {
    row["Incantation"]: build_incantation_pattern(row["Incantation"])
    for _, row in spells_df.iterrows()
}

# Collect every spell use
usage_rows = []

for _, row in dialogue.iterrows():
    text = normalize_for_matching(row["Dialogue Text"])
    char_name = row["Character Name"]
    char_id = row["Character ID"]

    for incantation, pattern in spell_patterns.items():
        hits = len(pattern.findall(text))
        if hits > 0:
            spell_row = spells_df.loc[spells_df["Incantation"] == incantation].iloc[0]
            usage_rows.append({
                "Character ID": char_id,
                "Character Name": char_name,
                "Spell Name": spell_row["Spell Name"],
                "Incantation": incantation,
                "Topic Label": spell_row["Topic Label"],
                "Topic_0": spell_row["Topic_0"],
                "Topic_1": spell_row["Topic_1"],
                "Topic_2": spell_row["Topic_2"],
                "Topic_3": spell_row["Topic_3"],
                "Uses": hits
            })

usage_df = pd.DataFrame(usage_rows)

# Total uses per character + spell
character_spell_counts = (
    usage_df.groupby(
        [
            "Character Name", "Spell Name", "Incantation", "Topic Label",
            "Topic_0", "Topic_1", "Topic_2", "Topic_3"
        ],
        as_index=False,
        dropna=False
    )["Uses"].sum()
)

# Unique spell count per character
unique_spell_counts = (
    usage_df.groupby("Character Name")["Spell Name"]
    .nunique()
    .reset_index(name="Unique Spells Used")
)

# Optional: total spell uses per character
total_spell_uses = (
    usage_df.groupby("Character Name")["Uses"]
    .sum()
    .reset_index(name="Total Spell Uses")
)

                         Spell Name       Incantation  Dominant Topic  \
0                   Summoning Charm             Accio               1   
1                Water-Making Spell         Aguamenti               3   
2  Launch an object up into the air  Alarte Ascendare               3   
3                   Unlocking Charm         Alohomora               2   
4            Spider repelling spell     Arania Exumai               0   

                      Topic Label   Topic_0   Topic_1   Topic_2   Topic_3  
0  LDA Topic 1: Defensive/Healing  0.037270  0.915852  0.021464  0.025414  
1     LDA Topic 3: Offensive/Dark  0.025800  0.018468  0.014858  0.940875  
2     LDA Topic 3: Offensive/Dark  0.019728  0.014122  0.011361  0.954789  
3    LDA Topic 2: Revealing/Other  0.037269  0.026678  0.910639  0.025414  
4    LDA Topic 0: Control/Utility  0.961064  0.014122  0.011361  0.013452  


In [4]:
#print(character_spell_counts.head(3))
#print(unique_spell_counts.head(3))
#print(total_spell_uses.head(20))
# Show spell uses for character ID 1 (Harry Potter): total spells used and unique spells used.
target_id = 1
target_name = character_map.get(target_id, "Unknown")

harry_spells = character_spell_counts[
    character_spell_counts["Character Name"] == target_name
].copy()

total_spells_used = int(harry_spells["Uses"].sum())
unique_spells_used = int(harry_spells["Spell Name"].nunique())

print(f"{target_name} (ID {target_id})")
print(f"Total spells used: {total_spells_used}")
print(f"Unique spells used: {unique_spells_used}")
# Give me a sorted list of his most used spells
harry_spells_sorted = harry_spells.sort_values(by="Uses", ascending=False)
print(harry_spells_sorted[["Spell Name", "Incantation", "Uses"]].head(10))

# Sum all spell counts across all characters and display it
total_spells = usage_df["Uses"].sum()
print(f"Total spells used across all characters: {total_spells}")


# Weighted probabilistic character-topic profiles
topic_cols = ["Topic_0", "Topic_1", "Topic_2", "Topic_3"]

for col in topic_cols:
    usage_df[col + "_weighted"] = usage_df[col] * usage_df["Uses"]

character_topic_profiles = (
    usage_df
    .groupby("Character Name")
    .apply(lambda g: pd.Series({
        col: g[col + "_weighted"].sum() / g["Uses"].sum()
        for col in topic_cols
    }))
    .reset_index()
)

character_topic_profiles["Dominant Topic"] = (
    character_topic_profiles[topic_cols].idxmax(axis=1)
)

topic_label_lookup = {
    "Topic_0": "Offensive/Dark",
    "Topic_1": "Defensive/Healing",
    "Topic_2": "Control/Utility",
    "Topic_3": "Revealing/Other"
}

character_topic_profiles["Character Profile"] = (
    character_topic_profiles["Dominant Topic"].map(topic_label_lookup)
)

print("Weighted character topic profiles:")
print(character_topic_profiles.head(10))

Harry Potter (ID 1)
Total spells used: 58
Unique spells used: 24
                Spell Name       Incantation  Uses
39     Wand-Lighting Charm             lumos    11
26          Patronus Charm  expecto patronum    10
25            Lumos Maxima      lumos maxima     5
35         Summoning Charm             accio     4
18         Disarming Charm      expelliarmus     3
34          Stunning Spell           stupefy     3
33  Spider repelling spell     arania exumai     2
31            Shield Charm           protego     2
29            Sectumsempra      sectumsempra     2
16          Blasting Curse         confringo     2
Total spells used across all characters: 138
Weighted character topic profiles:
        Character Name   Topic_0   Topic_1   Topic_2   Topic_3 Dominant Topic  \
0     Albus Dumbledore  0.021867  0.390812  0.197250  0.390070        Topic_1   
1                  All  0.015970  0.011431  0.961709  0.010890        Topic_2   
2  Bellatrix Lestrange  0.641332  0.326218  0.01485

C:\Users\Baxe\AppData\Local\Temp\ipykernel_36152\3209777013.py:34: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  usage_df


In [5]:
# Now we combine our spell-character mapping with the topic search from the spells notebook.
# From this, we get a probabilistic view of each character's spell-topic tendencies.
# - which topic of spells do they utilize more often?
# - combine topic probabilities from used spells
# - compute average topic distribution

# Aggregate directly from usage_df, which already includes each spell's topic label.
character_topic_usage = (
    usage_df.groupby(["Character Name", "Topic Label"], dropna=False, as_index=False)["Uses"]
    .sum()
)
character_topic_usage["Topic Label"] = character_topic_usage["Topic Label"].fillna("Unknown")
character_topic_usage["Probability"] = (
    character_topic_usage["Uses"]
    / character_topic_usage.groupby("Character Name")["Uses"].transform("sum")
)
character_topic_usage = character_topic_usage.sort_values(["Character Name", "Probability"], ascending=[True, False])

print("Character topic distributions computed and saved in character_topic_usage.")
print(character_topic_usage.head(10))

# If you still want Harry's distribution, filter it from the shared dataframe.
harry_topic_usage = character_topic_usage[character_topic_usage["Character Name"] == target_name].copy()
print(f"Topic distribution for {target_name}:")
print(harry_topic_usage[["Topic Label", "Probability"]])

Character topic distributions computed and saved in character_topic_usage.
        Character Name                     Topic Label  Uses  Probability
0     Albus Dumbledore  LDA Topic 1: Defensive/Healing     2          0.4
2     Albus Dumbledore     LDA Topic 3: Offensive/Dark     2          0.4
1     Albus Dumbledore    LDA Topic 2: Revealing/Other     1          0.2
3                  All    LDA Topic 2: Revealing/Other     2          1.0
4  Bellatrix Lestrange    LDA Topic 0: Control/Utility     2          1.0
5     Dolores Umbridge  LDA Topic 1: Defensive/Healing     1          0.5
6     Dolores Umbridge    LDA Topic 2: Revealing/Other     1          0.5
7         Draco Malfoy    LDA Topic 0: Control/Utility     1          1.0
8      Filius Flitwick    LDA Topic 0: Control/Utility     1          1.0
9         Fred Weasley  LDA Topic 1: Defensive/Healing     1          1.0
Topic distribution for Harry Potter:
                       Topic Label  Probability
15  LDA Topic 1: Defensive

In [6]:
first_four_ids = characters.sort_values("Character ID").head(4)[["Character ID", "Character Name"]]
first_four_character_names = first_four_ids["Character Name"].tolist()

first_four_probabilities = character_topic_usage[
    character_topic_usage["Character Name"].isin(first_four_character_names)
].merge(
    first_four_ids,
    on="Character Name",
    how="left"
).sort_values(["Character ID", "Probability"], ascending=[True, False])

print("Topic probabilities for the first 4 character IDs:")
print(first_four_probabilities[["Character ID", "Character Name", "Topic Label", "Uses", "Probability"]])

Topic probabilities for the first 4 character IDs:
    Character ID    Character Name                     Topic Label  Uses  \
3              1      Harry Potter  LDA Topic 1: Defensive/Healing    34   
4              1      Harry Potter    LDA Topic 0: Control/Utility    15   
5              1      Harry Potter     LDA Topic 3: Offensive/Dark     5   
6              1      Harry Potter    LDA Topic 2: Revealing/Other     4   
11             2       Ron Weasley    LDA Topic 2: Revealing/Other     3   
12             2       Ron Weasley    LDA Topic 0: Control/Utility     2   
13             2       Ron Weasley  LDA Topic 1: Defensive/Healing     1   
7              3  Hermione Granger    LDA Topic 0: Control/Utility    11   
8              3  Hermione Granger  LDA Topic 1: Defensive/Healing     7   
9              3  Hermione Granger    LDA Topic 2: Revealing/Other     5   
10             3  Hermione Granger     LDA Topic 3: Offensive/Dark     5   
0              4  Albus Dumbledore  L

In [7]:
usage_df.to_csv("character_spell_df.csv", index=False)
character_spell_counts.to_csv("character_spell_counts.csv", index=False)
character_topic_usage.to_csv("character_topic_usage.csv", index=False)
character_topic_profiles.to_csv("character_topic_profiles.csv", index=False)

print("Person 2 outputs saved.")

Person 2 outputs saved.
